# ตรวจจับและติดตามไฟบนวิดีโอ (YOLO26 + ByteTrack + Supervision)

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/th/Supervision_Video_Inferencing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🇹🇭 **ภาษาไทย** (เอกสารฉบับนี้) · [🇬🇧 English](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/Supervision_Video_Inferencing.ipynb)

รันโมเดลตรวจจับไฟจาก [`drone_fire_detection_yolo26.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb)
ไล่ไปทั้งวิดีโอ ติดตามผลตรวจจับแต่ละจุดข้ามเฟรม แล้วเขียนออกมาเป็นวิดีโอที่วาดผลลัพธ์ไว้แล้ว

**Runtime:** `Runtime` → `Change runtime type` → **T4 GPU** เพราะ inference วิดีโอบน CPU ช้ามาก

> **สิ่งที่เปลี่ยนไป** เมื่อก่อนการติดตามวัตถุต้องโคลน
> [ByteTrack](https://github.com/ifzhang/ByteTrack) มา build YOLOX จากซอร์ส แล้วติดตั้ง
> `onemetric` กับ `cython_bbox` ซึ่งเป็นชุดเครื่องมือที่ build บน Python รุ่นปัจจุบันไม่ผ่านแล้ว
> ทุกวันนี้ ByteTrack มากับ Ultralytics อยู่แล้ว `model.track(...)` จึงจัดการให้ครบโดยไม่ต้องลง
> อะไรเพิ่ม และ Supervision ก็อ่านหมายเลข track ออกมาจากผลลัพธ์ได้ตรง ๆ

## 1. ตรวจสอบ GPU

In [ ]:
!nvidia-smi

## 2. ติดตั้งไลบรารี

In [ ]:
%pip install -q "ultralytics>=8.4.122" "supervision>=0.30.0" "lap>=0.5.12"

import supervision as sv
import ultralytics

print("ultralytics:", ultralytics.__version__)
print("supervision:", sv.__version__)

## 3. ไฟล์ weights และวิดีโอต้นทาง

`best.pt` ไม่ได้เก็บไว้ในคลังโค้ด — ให้อัปโหลด checkpoint ที่ส่งออกมาจาก
[`drone_fire_detection_yolo26.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb)
ส่วนวิดีโอตัวอย่าง `fire.mp4` เซลล์ด้านล่างจะดึงมาจากคลังโค้ดให้เองอัตโนมัติ

In [ ]:
from pathlib import Path

MODEL_PATH = Path("best.pt")
SOURCE_VIDEO_PATH = Path("fire.mp4")
TARGET_VIDEO_PATH = Path("fire_result.mp4")

if not SOURCE_VIDEO_PATH.exists():
    !wget -q -O {SOURCE_VIDEO_PATH} https://github.com/jakkzz/Fire-Detection-Drone/raw/main/fire.mp4

if not MODEL_PATH.exists():
    print("ไม่พบ best.pt — อัปโหลดไฟล์ตอนนี้ได้เลย (หรือกลับไปรันโน้ตบุ๊กสำหรับเทรนก่อน)")
    try:
        from google.colab import files  # type: ignore

        files.upload()
    except ImportError:
        raise FileNotFoundError("วาง best.pt ไว้ในโฟลเดอร์เดียวกับโน้ตบุ๊กนี้")

video_info = sv.VideoInfo.from_video_path(str(SOURCE_VIDEO_PATH))
print(video_info)

## 4. โหลดโมเดล

In [ ]:
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))
model.fuse()

print("classes:", model.names)

## 5. ลองวาดผลลัพธ์บนเฟรมเดียวก่อน

เป็นการเช็กความเรียบร้อยแบบถูก ๆ ก่อนจะลงทุนรันยาวทั้งวิดีโอ

In [ ]:
import cv2

CONFIDENCE_THRESHOLD = 0.25

box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.6, text_thickness=2, text_padding=5)


def make_labels(detections: sv.Detections) -> list[str]:
    """สร้างป้ายกำกับรูปแบบ `#id class conf` โดยตัดหมายเลข id ออกเมื่อยังไม่ได้เปิดการติดตามวัตถุ"""
    labels = []
    for i in range(len(detections)):
        name = detections["class_name"][i]
        conf = detections.confidence[i]
        tracker_id = None if detections.tracker_id is None else detections.tracker_id[i]
        prefix = "" if tracker_id is None else f"#{tracker_id} "
        labels.append(f"{prefix}{name} {conf:.2f}")
    return labels


frame = next(sv.get_video_frames_generator(str(SOURCE_VIDEO_PATH)))

result = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)

annotated = box_annotator.annotate(scene=frame.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=make_labels(detections))

sv.plot_image(image=cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB), size=(10, 10))

## 6. ตรวจจับ + ติดตามวัตถุตลอดทั้งวิดีโอ

`model.track(..., persist=True, tracker="bytetrack.yaml")` จะเก็บสถานะของตัวติดตามไว้ระหว่าง
การเรียกแต่ละครั้ง ไฟแต่ละจุดจึงได้หมายเลข id ที่คงที่ข้ามเฟรม ส่วน
`sv.Detections.from_ultralytics` จะอ่านหมายเลขเหล่านั้นเข้ามาไว้ใน `detections.tracker_id`
แล้ว `TraceAnnotator` ก็เอาไปวาดเป็นเส้นร่องรอยการเคลื่อนที่

ByteTrack ถูกออกแบบมาให้กินผลตรวจจับที่ค่าความเชื่อมั่น *ต่ำ* เพราะมันดึงกรอบที่มั่นใจน้อย
กลับมาจับคู่กับ track ที่มีอยู่แล้ว ซึ่งเป็นที่มาของความแม่นยำส่วนใหญ่ของมัน โค้ดจึงส่งค่า `conf`
ต่ำ ๆ ให้ `track()` แล้วค่อยไปกรองค่าความเชื่อมั่นที่ผลลัพธ์ทีหลัง แทนที่จะไปอดอาหารตัวติดตาม
ตั้งแต่ขาเข้า

In [ ]:
from tqdm.auto import tqdm

trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=30)

TRACKER_INPUT_CONF = 0.1  # ตั้งต่ำโดยตั้งใจ: ByteTrack เอากรอบที่มั่นใจน้อยไปจับคู่กับ track ที่ยังวิ่งอยู่

frame_generator = sv.get_video_frames_generator(str(SOURCE_VIDEO_PATH))

with sv.VideoSink(str(TARGET_VIDEO_PATH), video_info) as sink:
    for frame in tqdm(frame_generator, total=video_info.total_frames):
        result = model.track(
            frame,
            conf=TRACKER_INPUT_CONF,
            persist=True,
            tracker="bytetrack.yaml",
            verbose=False,
        )[0]
        detections = sv.Detections.from_ultralytics(result)
        # ติดตามจากกรอบที่มั่นใจน้อยด้วย แต่วาดเฉพาะกรอบที่มั่นใจพอ
        detections = detections[detections.confidence >= CONFIDENCE_THRESHOLD]

        annotated = frame.copy()
        if detections.tracker_id is not None:
            annotated = trace_annotator.annotate(scene=annotated, detections=detections)
        annotated = box_annotator.annotate(scene=annotated, detections=detections)
        annotated = label_annotator.annotate(
            scene=annotated, detections=detections, labels=make_labels(detections)
        )

        sink.write_frame(annotated)

print("wrote:", TARGET_VIDEO_PATH.resolve())

## 7. เล่นวิดีโอผลลัพธ์ในหน้าโน้ตบุ๊ก

Supervision เขียนไฟล์ออกมาเป็น `mp4v` ซึ่งเบราว์เซอร์ถอดรหัสไม่ได้ ต้อง re-encode เป็น H.264
ด้วย ffmpeg (Colab ติดตั้งมาให้อยู่แล้ว) วิดีโอจึงจะเล่นในหน้าโน้ตบุ๊กได้

In [ ]:
PLAYABLE_VIDEO_PATH = Path("fire_result_h264.mp4")

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH} -vcodec libx264 -pix_fmt yuv420p {PLAYABLE_VIDEO_PATH}

print("wrote:", PLAYABLE_VIDEO_PATH.resolve())

In [ ]:
import base64

from IPython.display import HTML, display

encoded = base64.b64encode(PLAYABLE_VIDEO_PATH.read_bytes()).decode()
display(HTML(f'''
<video width="720" controls>
  <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
</video>
'''))

## 8. ดาวน์โหลดผลลัพธ์

In [ ]:
try:
    from google.colab import files  # type: ignore

    files.download(str(PLAYABLE_VIDEO_PATH))
except ImportError:
    print("ไม่ได้รันบน Colab ไฟล์ผลลัพธ์อยู่ที่:", PLAYABLE_VIDEO_PATH.resolve())